In [37]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import OneHotEncoder

from sklearn.ensemble import IsolationForest



import matplotlib.pyplot as plt

In [21]:
RANDOM_STATE = 777

In [22]:
df = pd.read_csv(
    filepath_or_buffer='car_price_dataset.csv'
)

In [23]:
df.head()

,Car_ID,Brand,Model_Year,Engine_Size,Fuel_Type,Transmission,Mileage,Doors,Owner_Count,Horsepower,Price
0,1,Ford,2023,1.2,Hybrid,Manual,180635,4,3,82,34309.25
1,2,Hyundai,2018,3.2,Electric,Manual,35628,2,4,259,55153.60
2,3,BMW,2008,2.2,Diesel,Manual,74672,3,2,333,41894.40
3,4,Hyundai,2017,2.2,Petrol,Automatic,51246,4,4,381,54046.70
4,5,Hyundai,2012,2.4,Electric,Manual,147233,3,4,290,38010.35


In [24]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Car_ID        2000 non-null   int64  
 1   Brand         2000 non-null   str    
 2   Model_Year    2000 non-null   int64  
 3   Engine_Size   2000 non-null   float64
 4   Fuel_Type     2000 non-null   str    
 5   Transmission  2000 non-null   str    
 6   Mileage       2000 non-null   int64  
 7   Doors         2000 non-null   int64  
 8   Owner_Count   2000 non-null   int64  
 9   Horsepower    2000 non-null   int64  
 10  Price         2000 non-null   float64
dtypes: float64(2), int64(6), str(3)
memory usage: 172.0 KB


In [25]:
df = df.rename(columns={"Mileage": "mileage_km"})

In [26]:
df.head()

,Car_ID,Brand,Model_Year,Engine_Size,Fuel_Type,Transmission,mileage_km,Doors,Owner_Count,Horsepower,Price
0,1,Ford,2023,1.2,Hybrid,Manual,180635,4,3,82,34309.25
1,2,Hyundai,2018,3.2,Electric,Manual,35628,2,4,259,55153.60
2,3,BMW,2008,2.2,Diesel,Manual,74672,3,2,333,41894.40
3,4,Hyundai,2017,2.2,Petrol,Automatic,51246,4,4,381,54046.70
4,5,Hyundai,2012,2.4,Electric,Manual,147233,3,4,290,38010.35


In [27]:
def standardize_columns_names(df: pd.DataFrame) -> pd.DataFrame:
    df_columns = df.columns
    mapper = {column: column.lower().replace(" ", "_") for column in df_columns}
    df = df.rename(columns=mapper)
    return df

df = standardize_columns_names(df=df)

In [28]:
df.head()

,car_id,brand,model_year,engine_size,fuel_type,transmission,mileage_km,doors,owner_count,horsepower,price
0,1,Ford,2023,1.2,Hybrid,Manual,180635,4,3,82,34309.25
1,2,Hyundai,2018,3.2,Electric,Manual,35628,2,4,259,55153.60
2,3,BMW,2008,2.2,Diesel,Manual,74672,3,2,333,41894.40
3,4,Hyundai,2017,2.2,Petrol,Automatic,51246,4,4,381,54046.70
4,5,Hyundai,2012,2.4,Electric,Manual,147233,3,4,290,38010.35


In [29]:
categorical_columns = df.select_dtypes(include=["str"]).columns.tolist()
numerical_columns = df.select_dtypes(include=["number"]).columns.tolist()

In [30]:
for column in categorical_columns:
    df[column] = (
        df[column]
        .str.lower()
        .str.replace(" ", "_")
    )

df.head()

,car_id,brand,model_year,engine_size,fuel_type,transmission,mileage_km,doors,owner_count,horsepower,price
0,1,ford,2023,1.2,hybrid,manual,180635,4,3,82,34309.25
1,2,hyundai,2018,3.2,electric,manual,35628,2,4,259,55153.60
2,3,bmw,2008,2.2,diesel,manual,74672,3,2,333,41894.40
3,4,hyundai,2017,2.2,petrol,automatic,51246,4,4,381,54046.70
4,5,hyundai,2012,2.4,electric,manual,147233,3,4,290,38010.35


### Detecting Anomalities with Isolation Forest

In [31]:
isolation_forest = IsolationForest(contamination=0.1, random_state=777)
isolation_forest.fit(df[numerical_columns])

df["anomaly"] = isolation_forest.predict(df[numerical_columns])

In [32]:
df.head()

,car_id,brand,model_year,engine_size,fuel_type,transmission,mileage_km,doors,owner_count,horsepower,price,anomaly
0,1,ford,2023,1.2,hybrid,manual,180635,4,3,82,34309.25,-1
1,2,hyundai,2018,3.2,electric,manual,35628,2,4,259,55153.60,1
2,3,bmw,2008,2.2,diesel,manual,74672,3,2,333,41894.40,1
3,4,hyundai,2017,2.2,petrol,automatic,51246,4,4,381,54046.70,-1
4,5,hyundai,2012,2.4,electric,manual,147233,3,4,290,38010.35,1


In [33]:
df

,car_id,brand,model_year,engine_size,fuel_type,transmission,mileage_km,doors,owner_count,horsepower,price,anomaly
0,1,ford,2023,1.2,hybrid,manual,180635,4,3,82,34309.25,-1
1,2,hyundai,2018,3.2,electric,manual,35628,2,4,259,55153.60,1
2,3,bmw,2008,2.2,diesel,manual,74672,3,2,333,41894.40,1
3,4,hyundai,2017,2.2,petrol,automatic,51246,4,4,381,54046.70,-1
4,5,hyundai,2012,2.4,electric,manual,147233,3,4,290,38010.35,1
...,...,...,...,...,...,...,...,...,...,...,...,...
1995,1996,hyundai,2023,4.5,diesel,manual,155033,2,3,310,61734.35,-1
1996,1997,toyota,2023,1.4,petrol,manual,25044,2,4,271,48467.80,-1
1997,1998,hyundai,2022,4.1,diesel,manual,104372,4,3,191,55714.40,1
1998,1999,bmw,2020,4.4,diesel,automatic,158047,4,1,186,53222.65,1


### Detecting Duplicates

In [36]:
df.duplicated().sum()

np.int64(0)

### Label Encoding

In [38]:
encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
encoded = encoder.fit_transform(df[categorical_columns])
encoded_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out(categorical_columns))

df = pd.concat([df, encoded_df], axis=1)

In [39]:
df.head()

,car_id,brand,model_year,engine_size,fuel_type,transmission,mileage_km,doors,owner_count,horsepower,...,brand_honda,brand_hyundai,brand_tesla,brand_toyota,fuel_type_diesel,fuel_type_electric,fuel_type_hybrid,fuel_type_petrol,transmission_automatic,transmission_manual
0,1,ford,2023,1.2,hybrid,manual,180635,4,3,82,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
1,2,hyundai,2018,3.2,electric,manual,35628,2,4,259,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
2,3,bmw,2008,2.2,diesel,manual,74672,3,2,333,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
3,4,hyundai,2017,2.2,petrol,automatic,51246,4,4,381,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
4,5,hyundai,2012,2.4,electric,manual,147233,3,4,290,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0


In [40]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 24 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   car_id                  2000 non-null   int64  
 1   brand                   2000 non-null   str    
 2   model_year              2000 non-null   int64  
 3   engine_size             2000 non-null   float64
 4   fuel_type               2000 non-null   str    
 5   transmission            2000 non-null   str    
 6   mileage_km              2000 non-null   int64  
 7   doors                   2000 non-null   int64  
 8   owner_count             2000 non-null   int64  
 9   horsepower              2000 non-null   int64  
 10  price                   2000 non-null   float64
 11  anomaly                 2000 non-null   int64  
 12  brand_bmw               2000 non-null   float64
 13  brand_ford              2000 non-null   float64
 14  brand_honda             2000 non-null   float64
 15

In [ ]:
df = df.drop(columns=categorical_columns)
df = df.drop(columns="anomaly")

ValueError: Need to specify at least one of 'labels', 'index' or 'columns'

In [42]:
df.head()

,car_id,model_year,engine_size,mileage_km,doors,owner_count,horsepower,price,anomaly,brand_bmw,...,brand_honda,brand_hyundai,brand_tesla,brand_toyota,fuel_type_diesel,fuel_type_electric,fuel_type_hybrid,fuel_type_petrol,transmission_automatic,transmission_manual
0,1,2023,1.2,180635,4,3,82,34309.25,-1,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
1,2,2018,3.2,35628,2,4,259,55153.60,1,0.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
2,3,2008,2.2,74672,3,2,333,41894.40,1,1.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
3,4,2017,2.2,51246,4,4,381,54046.70,-1,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
4,5,2012,2.4,147233,3,4,290,38010.35,1,0.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
